# 02 · Resultados

Lee las salidas de `scripts/train.py`, `scripts/tune.py`, `scripts/explain.py` y `scripts/leakage_ablation.py`. No entrena nada: los experimentos viven en scripts reproducibles, el notebook solo los interpreta.

In [ ]:
import sys, json
sys.path.insert(0, "../src")
import pandas as pd
from IPython.display import Image, Markdown, display
from lcrisk import config as C

res = json.loads((C.METRICS_DIR / "metrics.json").read_text())
display(Markdown((C.METRICS_DIR / "summary.md").read_text()))

## Ordenamiento y precisión sobre la clase minoritaria

In [ ]:
Image(C.FIGURES_DIR / "roc_pr.png")

## Calibración

Un modelo de admisión se usa para fijar tasas y límites, así que la probabilidad tiene que *significar* algo: si el modelo dice 20 %, ~20 % de esos clientes deberían hacer default.

In [ ]:
Image(C.FIGURES_DIR / "calibration.png")

## Tabla de deciles del mejor modelo

La pregunta de negocio: *si rechazo el X % de mayor score, ¿qué fracción de los defaults evito y a cuántos buenos clientes pierdo?*

In [ ]:
best = max(res["metrics"], key=lambda k: res["metrics"][k]["test"]["auc"])
print("Mejor modelo por AUC test:", best)
dec = pd.read_csv(C.METRICS_DIR / f"decile_{best}.csv")
dec.style.format({"default_rate": "{:.2%}", "cum_defaults_pct": "{:.1%}", "cum_goods_pct": "{:.1%}", "lift": "{:.2f}", "score_min": "{:.3f}", "score_max": "{:.3f}"}).background_gradient(subset=["default_rate"], cmap="Reds")

In [ ]:
display(Image(C.FIGURES_DIR / f"deciles_{best}.png")); display(Image(C.FIGURES_DIR / f"ks_{best}.png"))

## Umbral por costo

El umbral se eligió en **validación** minimizando `5·FN + 1·FP` y se aplicó sin tocar al test. Cambiar el ratio en `config.py` cambia la política de admisión; la tabla de deciles permite discutirlo con negocio.

In [ ]:
pd.DataFrame(res["thresholds"]).T

## Explicabilidad (SHAP)

In [ ]:
p = C.FIGURES_DIR / f"shap_summary_{best}.png"
display(Image(p)) if p.exists() else print(f"Ejecuta: python scripts/explain.py --model {best}")

## Verificación de fuga de información

Salida de `scripts/leakage_ablation.py`: mismo modelo, mismo split temporal; solo cambia si se permiten variables posteriores a la originación. Es la evidencia de por qué `config.LEAKAGE_COLUMNS` existe y de que un AUC muy alto en este dataset casi siempre significa leakage.

In [ ]:
p = C.METRICS_DIR / "leakage_ablation.md"
display(Markdown(p.read_text())) if p.exists() else print("Ejecuta: python scripts/leakage_ablation.py")